Import libraries ↓

In [ ]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

Data laden ↓

In [ ]:
df = pd.read_csv('trainingsdata.csv')

FEATURES = [
    'speler_x_midden', 'speler_x_links', 'ruimte_rechts',
    'steen_x', 'steen_y', 'steen_snelheid',
    'relatief_verschil', 'aantal_stenen', 'score'
]

X = df[FEATURES].values
y = df['actie'].values + 1

print(f'Rijen geladen: {len(df)}')

Normaliseren en splitsen ↓

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42
)

print(f'Trainingsset: {len(X_train)} rijen')
print(f'Testset:      {len(X_test)} rijen')

Model bouwen ↓

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(3,   activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Model trainen ↓

In [ ]:
klassen = np.unique(y_train)
gewichten = compute_class_weight('balanced', classes=klassen, y=y_train)
class_weight_dict = dict(zip(klassen, gewichten))

model.fit(X_train, y_train, epochs=15, batch_size=128, validation_split=0.1, class_weight=class_weight_dict)

Evaluatie ↓

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Accuracy: {accuracy * 100:.1f}%')
print(f'Loss:     {loss:.4f}')
print()

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(
    y_test, y_pred,
    target_names=['Links', 'Stil', 'Rechts'],
    zero_division=0
))

Model en scaler opslaan ↓

In [ ]:
model.save('model_keras.keras')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model opgeslagen als model_keras.keras')
print('Scaler opgeslagen als scaler.pkl')
print('Klaar! Voer nu ai_speler.py uit.')